**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Linear Algebra for Signals

The most-borrowed-from course in the curriculum: transforms are changes of basis, filters and networks are matrices, PCA and attention are eigen/SVD stories. Five sessions build exactly the linear algebra the other workshops silently assume — always with a signal in hand.

## 0. Introduction

The through-line: **a signal is a vector; every operation we care about is a matrix; understanding a matrix means finding the basis in which it is simple.**

## 1. Pre-requisites

[Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) (NumPy). No prior linear algebra assumed; comfort with vectors as arrows helps.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Vectors, Bases & Change of Basis* (~35 min)
**Goal:** treat signals as vectors; understand span, independence, and what a basis buys you.
**Feeds into:** Session 2 (projections).

---

## 2. Signals Are Vectors

💡 **Intuition.** A sampled signal of length $N$ *is* a point in $\mathbb{R}^N$ — each sample is a coordinate. Adding signals, scaling signals, mixing signals: all vector operations. A **basis** is a set of $N$ independent reference signals; *any* signal is a unique recipe (coordinate vector) over them. Changing basis doesn't change the signal — it changes which recipe you read.

**Definitions.** Vectors $\{v_1, \dots, v_k\}$ are *linearly independent* if $\sum_i c_i v_i = 0$ forces all $c_i = 0$ (no vector is a mix of the others). Their *span* is everything expressible as such mixes. A *basis* of $\mathbb{R}^N$ = $N$ independent vectors; then every $x$ has unique coordinates. If the basis vectors are orthonormal ($B^T B = I$), coordinates are just inner products: $c = B^T x$.

In [2]:
N = 64
t = np.arange(N)

# Basis 1: the standard basis (each vector = one sample spike)
# Basis 2: cosines of increasing frequency (a mini-DCT) — orthonormal
k = np.arange(N)
B = np.cos(np.pi * (t[:, None] + 0.5) * k[None, :] / N)
B[:, 0] *= 1 / np.sqrt(2)
B *= np.sqrt(2 / N)
print("orthonormal check ‖BᵀB − I‖ =", np.abs(B.T @ B - np.eye(N)).max().round(12))

orthonormal check ‖BᵀB − I‖ = 0.0


In [3]:
# The SAME smooth signal, read in both bases
x = np.exp(-0.5 * ((t - 24) / 8) ** 2)          # a smooth bump
c = B.T @ x                                      # coordinates in the cosine basis

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].plot(x); axes[0].set_title("standard-basis coordinates (the samples)")
axes[1].stem(c); axes[1].set_title("cosine-basis coordinates: nearly all zero!")
plt.tight_layout(); plt.show()
print(f"samples needed to capture 99.9% energy: standard {N}, cosine {int((np.sort(c**2)[::-1].cumsum() / (c**2).sum() < 0.999).sum()) + 1}")

samples needed to capture 99.9% energy: standard 64, cosine 6


/tmp/ipykernel_2010461/3565037298.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


That collapse — smooth signal, sparse cosine recipe — is why JPEG exists, and it's the entire premise of [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb). The DFT of [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) is exactly a change to a complex-exponential basis.

---
### 🕐 Session 2 of 5 — *Projections & Least Squares* (~35 min)
**Goal:** project onto subspaces; derive the normal equations; fit models as projections.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (eigendecomposition).

---

## 3. Projection

💡 **Intuition.** If your signal can't be built exactly from the vectors you have, the best you can do is the **shadow**: the point of the subspace closest to the signal. The defining property is that the leftover (residual) is *orthogonal* to the subspace — if it weren't, you could slide within the subspace to shrink the error. Least squares is nothing but this geometry with data.

**Normal equations.** To minimize $\|Ac - x\|^2$ over recipes $c$ (columns of $A$ = your building blocks): the residual must satisfy $A^T (x - Ac) = 0$, giving

$$A^T A \, c = A^T x \qquad \Rightarrow \qquad \hat{x} = A (A^T A)^{-1} A^T x$$

The matrix $P = A(A^TA)^{-1}A^T$ is the *projector* onto the column space: $P^2 = P$ (projecting twice changes nothing).

In [4]:
# Denoise by projection: noisy smooth signal onto the first 8 cosine basis vectors
x_clean = np.exp(-0.5 * ((t - 24) / 8) ** 2)
x_noisy = x_clean + 0.1 * rng.standard_normal(N)

A = B[:, :8]                                    # low-frequency subspace
xhat = A @ (A.T @ x_noisy)                      # orthonormal columns ⇒ projector is AAᵀ

plt.figure(figsize=(8, 2.6))
plt.plot(x_noisy, alpha=0.5, label="noisy")
plt.plot(xhat, linewidth=2, label="projection onto 8-D smooth subspace")
plt.plot(x_clean, "k--", linewidth=1, label="truth")
plt.legend(); plt.title("Denoising = projection")
plt.tight_layout(); plt.show()
print(f"noise RMSE {np.std(x_noisy - x_clean):.4f} → projected RMSE {np.std(xhat - x_clean):.4f}")

noise RMSE 0.0912 → projected RMSE 0.0453


/tmp/ipykernel_2010461/1909784845.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# Least squares as model fitting: recover a line + sine trend from noisy data
m = 200
tt = np.linspace(0, 1, m)
y = 2.0 + 1.5 * tt + 0.8 * np.sin(2 * np.pi * 3 * tt) + 0.3 * rng.standard_normal(m)

A = np.stack([np.ones(m), tt, np.sin(2 * np.pi * 3 * tt), np.cos(2 * np.pi * 3 * tt)], axis=1)
c, *_ = np.linalg.lstsq(A, y, rcond=None)
print("recovered coefficients:", c.round(3), " (truth: [2, 1.5, 0.8, 0])")

recovered coefficients: [2.022 1.439 0.805 0.029]  (truth: [2, 1.5, 0.8, 0])


The Wiener solution $\mathbf{w}_o = R^{-1}\mathbf{p}$ in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) *is* a normal equation — now you know why it looks the way it does.

---
### 🕐 Session 3 of 5 — *Eigendecomposition* (~35 min)
**Goal:** find the directions a matrix merely stretches; diagonalize symmetric matrices.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (SVD).

---

## 4. Eigenvectors

💡 **Intuition.** Most vectors get rotated *and* stretched by a matrix. Eigenvectors are the exceptions — directions the matrix only **stretches** ($Av = \lambda v$). They are the matrix's natural axes: in the eigenbasis, the matrix becomes a diagonal list of stretch factors, and hard things (powers, exponentials, stability) become arithmetic. For symmetric matrices the spectral theorem is the jackpot: eigenvalues real, eigenvectors orthonormal — a *perfect* basis.

**Spectral theorem.** Symmetric $S = S^T \in \mathbb{R}^{N\times N}$ has $S = Q \Lambda Q^T$ with $Q$ orthonormal, $\Lambda$ real diagonal. Consequences: $S^k = Q\Lambda^k Q^T$; quadratic form $x^T S x = \sum_i \lambda_i (q_i^T x)^2$; positive-definite ⇔ all $\lambda_i > 0$ (the loss *bowl* of the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) is such a form).

In [6]:
# Power iteration: the eigenvector algorithm you can write in four lines
S = np.array([[2.0, 1.0], [1.0, 3.0]])
v = rng.standard_normal(2)
for _ in range(50):
    v = S @ v
    v /= np.linalg.norm(v)
lam = v @ S @ v
w, V = np.linalg.eigh(S)
print(f"power iteration: λ={lam:.6f}  v={np.round(v, 4)}")
print(f"numpy eigh:      λ={w[-1]:.6f}  v={np.round(V[:, -1], 4)}   (sign may flip)")

power iteration: λ=3.618034  v=[0.5257 0.8507]
numpy eigh:      λ=3.618034  v=[0.5257 0.8507]   (sign may flip)


In [7]:
# Eigenvectors of a covariance = the axes of the data cloud
X = rng.standard_normal((500, 2)) @ np.array([[2.0, 0.0], [1.2, 0.5]])
C = np.cov(X.T)
w, V = np.linalg.eigh(C)

plt.figure(figsize=(4.5, 4.5))
plt.scatter(*X.T, s=4, alpha=0.4)
for lam_i, vec in zip(w, V.T):
    plt.arrow(0, 0, *(2 * np.sqrt(lam_i) * vec), width=0.03, color="crimson")
plt.axis("equal"); plt.title("covariance eigenvectors = the cloud's own axes")
plt.tight_layout(); plt.show()
print("eigenvalues (variances along the axes):", w.round(2))

eigenvalues (variances along the axes): [0.18 5.09]


/tmp/ipykernel_2010461/423711697.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Convergence of LMS in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is governed by the eigenvalues of exactly this matrix $R$ — the $\mu < 2/\lambda_{max}$ bound is now a picture: the steepest axis of the bowl sets the speed limit.

---
### 🕐 Session 4 of 5 — *The SVD* (~40 min)
**Goal:** the decomposition that works for EVERY matrix; low-rank approximation and PCA.
**Builds on:** Session 3. &nbsp; **Feeds into:** Session 5 (matrix calculus).

---

## 5. Singular Value Decomposition

💡 **Intuition.** Eigendecomposition needs square (ideally symmetric) matrices. The SVD works for **any** matrix: $A = U\Sigma V^T$ says every linear map is *rotate → stretch along axes → rotate* — no exceptions. The singular values in $\Sigma$ rank the map's actions by importance, and chopping the small ones gives the **best possible** low-rank approximation (Eckart–Young). PCA, compression, denoising, and pseudo-inverses are all this one move.

In [8]:
# Low-rank approximation of a structured "image"
img = np.zeros((64, 64))
img[10:54, 10:54] = 1.0
yy, xx = np.mgrid[0:64, 0:64]
img += 0.5 * np.sin(2 * np.pi * xx / 16) * (yy > 32)
img += 0.05 * rng.standard_normal((64, 64))

U, s, Vt = np.linalg.svd(img)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("original (rank 64)")
for ax, r in zip(axes[1:], [1, 3, 8]):
    approx = U[:, :r] @ np.diag(s[:r]) @ Vt[:r]
    ax.imshow(approx, cmap="gray"); ax.set_title(f"rank {r}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("energy in top 8 of 64 singular values:", f"{(s[:8]**2).sum() / (s**2).sum():.1%}")

energy in top 8 of 64 singular values: 99.7%


/tmp/ipykernel_2010461/479179373.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [9]:
# PCA = SVD of centered data (relating S3 and S4)
Xc = X - X.mean(0)
U2, s2, Vt2 = np.linalg.svd(Xc, full_matrices=False)
print("PCA directions from SVD:\n", np.round(Vt2.T, 4))
print("same axes as covariance eigh (up to sign/order):\n", np.round(V, 4))
print("singular values² / (n−1) =", np.round(s2**2 / (len(Xc) - 1), 2), " vs eigenvalues", np.round(w[::-1], 2))

PCA directions from SVD:
 [[ 0.9939 -0.1105]
 [ 0.1105  0.9939]]
same axes as covariance eigh (up to sign/order):
 [[ 0.1105 -0.9939]
 [-0.9939 -0.1105]]
singular values² / (n−1) = [5.09 0.18]  vs eigenvalues [5.09 0.18]


---
### 🕐 Session 5 of 5 — *Matrix Calculus* (~35 min)
**Goal:** differentiate through vectors and matrices — the notation backprop is written in.
**Builds on:** Sessions 2–4.

---

## 6. Gradients of Vector Functions

💡 **Intuition.** Matrix calculus is scalar calculus plus bookkeeping: the gradient of a scalar with respect to a vector is just the vector of partials, and the two identities below cover 90% of everything in this curriculum. When in doubt, *check numerically* — the habit the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) already taught you.

**The two workhorses** (with $S$ symmetric):

$$\nabla_x \, (a^T x) = a \qquad \nabla_x \, (x^T S x) = 2 S x$$

**Application — least squares in one line.** $J(c) = \|Ac - x\|^2 = c^T A^T A c - 2 x^T A c + x^T x$, so $\nabla_c J = 2 A^T A c - 2 A^T x$. Set to zero → the normal equations of Session 2. The LMS update of [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is gradient descent on exactly this $J$.

In [10]:
# Verify both identities numerically — never trust a derivative you haven't checked
def num_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = eps
        g[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g

x0 = rng.standard_normal(5)
a = rng.standard_normal(5)
M = rng.standard_normal((5, 5)); S5 = M + M.T

print("‖∇(aᵀx) − a‖          =", np.abs(num_grad(lambda x: a @ x, x0) - a).max().round(9))
print("‖∇(xᵀSx) − 2Sx‖       =", np.abs(num_grad(lambda x: x @ S5 @ x, x0) - 2 * S5 @ x0).max().round(6))

A2 = rng.standard_normal((8, 5)); b2 = rng.standard_normal(8)
g_analytic = 2 * A2.T @ (A2 @ x0 - b2)
g_numeric  = num_grad(lambda c: np.sum((A2 @ c - b2)**2), x0)
print("‖∇‖Ac−b‖² check‖      =", np.abs(g_analytic - g_numeric).max().round(6))

‖∇(aᵀx) − a‖          = 0.0
‖∇(xᵀSx) − 2Sx‖       = 0.0
‖∇‖Ac−b‖² check‖      = 0.0


## 7. Conclusion

Signals are vectors; bases make hard signals sparse; projections are optimal approximations; eigenvectors are a matrix's natural axes; the SVD does it for every matrix; and two gradient identities unlock all the optimization. Every other workshop now has its algebra.

---
## Where next

- [Optimization](../Optimization/Optimization.ipynb) — what to *do* with those gradients.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — the Fourier basis as the star change-of-basis.
- [Statistical Signal Processing](../../Intro_DSP/Statistical_Signal_Processing.ipynb) — covariance eigenstructure at work.